# BMP Data Cleaning: Iowa NRS HUC-8 Practice Adoption

Cleans the **per-watershed conservation-practice adoption** table from the Iowa
Nutrient Reduction Strategy (INRS) into a tidy, type-safe table keyed on
`huc8_code` + `year` + `practice_type` + `unit`, ready to aggregate as a
watershed-scale mitigation control variable.

**Input:**  `data/tabular/01_raw/bmp/iowa-nrs-bmp-huc8.csv`
**Output:** `data/tabular/02_clean/bmp/iowa-nrs-bmp-huc8-clean.csv`

Each raw row reports the cumulative adoption of one BMP `practice_type`
(cover crops, bioreactors / saturated buffers, CREP wetlands, erosion control)
in one HUC-8 watershed for one `year`, measured either as an `Acres` footprint
or a `Number` count. The companion `iowa-nrs-tracking-clean.ipynb` cleans the
much wider full INRS export.

**Pipeline**
1. Load the raw extract (ids as strings).
2. **Fix types & the join key** — strip thousands separators from `value` and
   coerce to float, coerce `year` to a nullable int, and **zero-pad `huc8_code`
   to the canonical 8-digit string** (reading it as an integer silently dropped
   the leading zero from codes like `07080105`).
3. **Drop the constant column** — `assessment` is `"HUC8 Practice Adoption"` on
   every row and carries no signal.
4. **De-duplicate & collapse the key** — drop exact duplicate rows (the same
   record re-exported with `"1,970.00"` vs `1970` formatting), then enforce one
   value per `(huc8_code, year, practice_type, category, unit)` by taking the
   `max` where a few keys still disagree.
5. **Range-validate** — null negative counts/areas (adoption can't be negative).
6. Sanity-check and save.

> **Why `category` stays in the key:** `bioreactor_sat_buffer` is reported under
> both `"Bioreactors and Saturated Buffers"` and an
> `"... (Updated 2022)"` revision that **overlaps the same years** with
> different figures — they are two distinct accountings, not duplicates, so
> collapsing across `category` would conflate them.

> **Path note:** like the other migrated cleaners this reads `01_raw` and writes
> `02_clean`. No BMP merge step exists yet; when one is added it should read from
> `02_clean/bmp/` and join on the zero-padded `huc8_code`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "bmp"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "bmp"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/bmp
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/bmp


## Step 1 — Load

~2.2K rows. We read `huc8_code` and `value` as strings up front: `huc8_code` is
a categorical identifier (never arithmetic, and leading-zero-sensitive), and
`value` carries thousands separators that would otherwise break a naive numeric
read.

In [2]:
df = pd.read_csv(
    RAW_DIR / "iowa-nrs-bmp-huc8.csv",
    dtype={"huc8_code": "string", "value": "string"},
)
n_raw = len(df)
print(f"Loaded {n_raw:,} rows")
print("Columns:", list(df.columns))
df.head()

Loaded 2,195 rows
Columns: ['year', 'practice_type', 'category', 'assessment', 'value', 'unit', 'huc8_code', 'huc8_name']


,year,practice_type,category,assessment,value,unit,huc8_code,huc8_name
0,2019,bioreactor_sat_buffer,Bioreactors and Saturated Buffers,HUC8 Practice Adoption,2,Number,10170204,Rock
1,2020,bioreactor_sat_buffer,Bioreactors and Saturated Buffers,HUC8 Practice Adoption,2,Number,10170204,Rock
2,2021,bioreactor_sat_buffer,Bioreactors and Saturated Buffers,HUC8 Practice Adoption,2,Number,10170204,Rock
3,2022,bioreactor_sat_buffer,Bioreactors and Saturated Buffers,HUC8 Practice Adoption,2,Number,10170204,Rock
4,2019,bioreactor_sat_buffer,Bioreactors and Saturated Buffers (Updated 2022),HUC8 Practice Adoption,4,Number,10230002,Floyd


## Step 2 — Fix types & the join key

Strip the `,` thousands separators and coerce `value` to float; coerce `year` to
a nullable integer; and **zero-pad `huc8_code` to 8 characters**. The raw file
stored HUC-8 codes as integers, which dropped the leading zero from every
`07……` watershed (so `7080105` should be `07080105`) — re-padding restores the
canonical key the spatial layers and other tabular sources use.

In [3]:
df["value"] = pd.to_numeric(
    df["value"].str.replace(",", "", regex=False), errors="coerce"
)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

# HUC-8 codes are 8-digit strings; integer storage dropped the leading zero.
df["huc8_code"] = df["huc8_code"].str.strip().str.zfill(8)

for col in ("practice_type", "category", "unit", "huc8_name"):
    df[col] = df[col].str.strip()

print("value NaNs after parse:", int(df["value"].isna().sum()))
print("year  NaNs after parse:", int(df["year"].isna().sum()))
print("huc8_code lengths:", sorted(df["huc8_code"].str.len().unique()))
print("huc8_code <-> huc8_name 1:1:",
      (df.groupby("huc8_code")["huc8_name"].nunique() <= 1).all())
df[["year", "practice_type", "category", "unit", "value", "huc8_code", "huc8_name"]].head()

value NaNs after parse: 0
year  NaNs after parse: 0
huc8_code lengths: [np.int64(8)]
huc8_code <-> huc8_name 1:1: True


,year,practice_type,category,unit,value,huc8_code,huc8_name
0,2019,bioreactor_sat_buffer,Bioreactors and Saturated Buffers,Number,2.0,10170204,Rock
1,2020,bioreactor_sat_buffer,Bioreactors and Saturated Buffers,Number,2.0,10170204,Rock
2,2021,bioreactor_sat_buffer,Bioreactors and Saturated Buffers,Number,2.0,10170204,Rock
3,2022,bioreactor_sat_buffer,Bioreactors and Saturated Buffers,Number,2.0,10170204,Rock
4,2019,bioreactor_sat_buffer,Bioreactors and Saturated Buffers (Updated 2022),Number,4.0,10230002,Floyd


## Step 3 — Drop the constant column

`assessment` is `"HUC8 Practice Adoption"` for every row, so it adds no
information to a HUC-8 adoption table. We drop it.

In [4]:
assert df["assessment"].nunique() == 1, df["assessment"].unique()
print("Dropping constant column 'assessment' =", repr(df["assessment"].iloc[0]))
df = df.drop(columns="assessment")

Dropping constant column 'assessment' = 'HUC8 Practice Adoption'


## Step 4 — De-duplicate & collapse the join key

First drop **exact** duplicate rows — the export repeats records with cosmetic
differences only (e.g. `"1,970.00"` and `1970`, identical once parsed). A
handful of keys still carry conflicting figures across re-pulls; since these are
cumulative adoption counts/areas we keep the **largest** (most complete) value
per `(huc8_code, year, practice_type, category, unit)`, which also yields a key
the downstream merge can't silently fan out on.

In [5]:
KEY = ["huc8_code", "year", "practice_type", "category", "unit"]

before = len(df)
df = df.drop_duplicates()
print(f"Exact-duplicate rows dropped: {before - len(df):,}  ({before:,} -> {len(df):,})")

n_conflict = df.duplicated(KEY, keep=False).sum()
print(f"Rows in conflicting key groups (differing value): {int(n_conflict):,}")
df = (
    df.sort_values("value")
      .groupby(KEY, as_index=False, dropna=False)
      .agg({"value": "max", "huc8_name": "first"})
)
assert not df.duplicated(KEY).any()
print(f"Rows after collapsing to one value per key: {len(df):,}")

Exact-duplicate rows dropped: 416  (2,195 -> 1,779)
Rows in conflicting key groups (differing value): 160
Rows after collapsing to one value per key: 1,699


## Step 5 — Range validation

Practice adoption is a non-negative count (`Number`) or footprint (`Acres`).
Null anything negative — it can only be a sentinel or decoding error, not a real
observation. (None are present today; this guards future re-pulls.)

In [6]:
bad = df["value"].notna() & (df["value"] < 0)
print(f"Negative values nulled: {int(bad.sum())}")
df.loc[bad, "value"] = np.nan

Negative values nulled: 0


## Step 6 — Tidy column order & sanity check

Order the table key-first, confirm the join key is unique, and summarise
coverage by practice and unit.

In [7]:
OUTPUT_COLS = ["huc8_code", "huc8_name", "year", "practice_type", "category", "unit", "value"]
df = df[OUTPUT_COLS].sort_values(["huc8_code", "practice_type", "unit", "year"]).reset_index(drop=True)

assert not df.duplicated(["huc8_code", "year", "practice_type", "category", "unit"]).any()
print(f"Rows: {len(df):,} ({len(df) / n_raw:.0%} of raw)  |  "
      f"HUC-8s: {df['huc8_code'].nunique()}  |  "
      f"years: {int(df['year'].min())}-{int(df['year'].max())}")
print(f"value missing: {int(df['value'].isna().sum())}\n")
print("Coverage by practice_type x unit:")
print(df.groupby(["practice_type", "unit"])["value"].agg(["count", "min", "max"]).round(1))
df.head()

Rows: 1,699 (77% of raw)  |  HUC-8s: 56  |  years: 2003-2022
value missing: 0

Coverage by practice_type x unit:
                              count    min      max
practice_type         unit                         
bioreactor_sat_buffer Number    369    1.0    103.0
cover_crop            Acres     112  111.2  99400.0
crep_wetland          Acres     291  514.0  28700.0
                      Number    295    1.0     20.0
erosion_control       Acres     632   10.0  23000.0


,huc8_code,huc8_name,year,practice_type,category,unit,value
0,07020009,Blue Earth,2017,cover_crop,340,Acres,2714.68
1,07020009,Blue Earth,2022,cover_crop,340,Acres,5020.0
2,07040008,Root,2017,cover_crop,340,Acres,111.22
3,07040008,Root,2022,cover_crop,340,Acres,160.0
4,07060001,Coon-Yellow,2018,bioreactor_sat_buffer,Bioreactors and Saturated Buffers (Updated 2022),Number,1.0


## Step 7 — Save

In [8]:
out_file = CLEAN_DIR / "iowa-nrs-bmp-huc8-clean.csv"
df.to_csv(out_file, index=False)
print(f"Saved {len(df):,} rows -> {out_file}")

Saved 1,699 rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/bmp/iowa-nrs-bmp-huc8-clean.csv
